In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.use_deterministic_algorithms(True, warn_only=True)

In [2]:
!uv pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=6d5726a2-89fc-4b55-b722-6a9ec1ffddac
To: /kaggle/working/dataset.zip
100%|████████████████████████████████████████| 356M/356M [00:11<00:00, 31.0MB/s]


In [3]:
%pip install comet_ml -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import sys

# Keep Comet tracking enabled, but hide INFO-level console messages.
os.environ["COMET_LOGGING_CONSOLE"] = "ERROR"

sys.path.append('/kaggle/input/datasets/maksimbessolitsyn/yambdadataset')
sys.path.append('/kaggle/input/models/maksimbessolitsyn/sasrec/pytorch/default/35')

In [ ]:
import logging
import warnings

import comet_ml
import torch
from torch import nn

# Silence torch warnings/noisy logs, keep tqdm untouched.
warnings.filterwarnings("ignore", category=UserWarning, module=r"torch(\.|$)")
warnings.filterwarnings("ignore", category=FutureWarning, module=r"torch(\.|$)")
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)

# Suppress Comet INFO messages in notebook output.
logging.getLogger("comet_ml").setLevel(logging.ERROR)

from train import run_ddp_training, ExperimentConfig
from config import COMET_API_KEY

In [6]:
comet_ml.login(api_key=COMET_API_KEY)

In [ ]:
experiment = ExperimentConfig(
    graph=ExperimentConfig.GraphConfig(
        vocab_size=157_162,
        max_seq_len=100,
        n_layers=4,
        d_model=256,
        n_heads=4,
    ),
    tau=ExperimentConfig.TauConfig(
        type="ConstantTau",
        initial_tau=0.45,
        tau_min=0.05,
        tau_max=0.045,
        num_epochs=5,
        num_tokens_per_epoch=4_019_032
    ),
    training_dataset=ExperimentConfig.TrainingDatasetConfig(
        batch_size=32,
        device="cuda",
        chunk_rows=64000,
        shuffle=True,
        seed=42, 
        pin_memory=True,
        vocab_size=157_162,
        uniform_negative_items=30_000, 
        in_batch_negative_items=1_000, 
    ),
    test_dataset=ExperimentConfig.TestDatasetConfig(
        batch_size=32,
        device="cuda",
    ),
    optimizer=ExperimentConfig.OptimizerConfig(
        lr=1e-3,
        weight_decay=1e-5,
    ),
    scheduler=ExperimentConfig.SchedulerConfig(
        type=None,
    ),
    training=ExperimentConfig.TrainingConfig(
        num_epochs=15,
        grad_clip=1.0, 
        eval_every=1, 
        logging=True,
    ),
    evaluator=ExperimentConfig.EvaluatorConfig(
        topk=100
    ),
    vocab_size=157_162, # divisible by 16 for efficient GPU usage
    max_seq_len=100,
    bos=0,
)

In [ ]:
for tau in [0.04, 0.045, 0.05, 0.055]:
    run_ddp_training(experiment._replace(tau=ExperimentConfig.TauConfig(
        type="constant",
        initial_tau=tau,
        tau_min=None,
        tau_max=None,
        num_epochs=None,
        num_tokens_per_epoch=None
    )))

100%|██████████| 293/293 [00:10<00:00, 28.06it/s]


{'hitrate': 0.3410778187256316, 'recall': 0.11395944209807399, 'ndcg': 0.0452254604649848, 'coverage': 0.3104093358870429} 0.04


100%|██████████| 293/293 [00:10<00:00, 28.08it/s]


{'hitrate': 0.3512524702237889, 'recall': 0.11949002806893694, 'ndcg': 0.049056676368799776, 'coverage': 0.3004384150880966} 0.045


100%|██████████| 293/293 [00:10<00:00, 28.02it/s]


{'hitrate': 0.3466591892324948, 'recall': 0.1180767160146886, 'ndcg': 0.04742776990194541, 'coverage': 0.25817494607303526} 0.05


100%|██████████| 293/293 [00:10<00:00, 28.12it/s]


{'hitrate': 0.34075735726112266, 'recall': 0.1160526750023391, 'ndcg': 0.04637995821432303, 'coverage': 0.24819129914671315} 0.055


In [15]:
print(
    run_experiment(ParameterTau(initial_tau=0.05), num_epochs=15), 
    tau,
)

epoch 0:   0%|          | 0/1255 [00:00<?, ?it/s]/kaggle/input/models/maksimbessolitsyn/sasrec/pytorch/default/35/model/model/graph.py:74: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  writer.log_metric("train/tau", value=float(tau.mean()), step=step) if writer is not None else None
100%|██████████| 293/293 [00:10<00:00, 28.20it/s]


{'hitrate': 0.2800299097366875, 'recall': 0.086511709784989, 'ndcg': 0.03249077469759841, 'coverage': 0.5406186170517381} 0.055


In [16]:
cases = [
    (0.035, 0.055),
    (0.04, 0.055),
    (0.04, 0.06),
]

In [17]:
for tau_min, tau_max in cases:
    print(
        run_experiment(
            LinearTau(tau_min=tau_min,
                      tau_max=tau_max,
                      num_epochs=15,
                     ),
            num_epochs=15
        ),
        tau_min, 
        tau_max, 
    )

100%|██████████| 293/293 [00:10<00:00, 27.92it/s]


{'hitrate': 0.3397692677455536, 'recall': 0.11381299216299962, 'ndcg': 0.04594093240308334, 'coverage': 0.32136016849392646} 0.035 0.055


100%|██████████| 293/293 [00:10<00:00, 28.10it/s]


{'hitrate': 0.34342786946536347, 'recall': 0.11501733899875934, 'ndcg': 0.04593255361812432, 'coverage': 0.30545887233785324} 0.04 0.055


100%|██████████| 293/293 [00:10<00:00, 28.16it/s]


{'hitrate': 0.3453506382524168, 'recall': 0.11604467790558083, 'ndcg': 0.04634533857284265, 'coverage': 0.28573973796903734} 0.04 0.06


In [18]:
for tau_min, tau_max in cases:
    print(
        run_experiment(
            CosTau(tau_min=tau_min,
                   tau_max=tau_max,
                   num_epochs=15,
                  ),
            num_epochs=15
        ),
        tau_min, 
        tau_max, 
    )

100%|██████████| 293/293 [00:10<00:00, 27.94it/s]


{'hitrate': 0.3311168082038135, 'recall': 0.10939373648669035, 'ndcg': 0.04318136733914769, 'coverage': 0.3578141603619311} 0.035 0.055


100%|██████████| 293/293 [00:10<00:00, 28.11it/s]


{'hitrate': 0.34708647118517333, 'recall': 0.11795546164763539, 'ndcg': 0.04714180459175646, 'coverage': 0.3407356974235955} 0.04 0.055


100%|██████████| 293/293 [00:10<00:00, 27.91it/s]


{'hitrate': 0.33581690968327726, 'recall': 0.11121564910118957, 'ndcg': 0.04337206163011018, 'coverage': 0.29287909542686613} 0.04 0.06
